Purpose of this notebook is to Merge Asset Impact Data with GOGET Data

See [Asset Impact Data Parse 2024 12 09.ipynb ](https://colab.research.google.com/drive/1m_UD1GSsmFjzyObIlEgAHJSlLM3wrwU9?usp=sharing) for parsing of Asset Impact Data

And see [Asset Impact Data Integration Workplan](https://docs.google.com/document/d/18ErkkB0Bw8ngkf2gnnuxptHNJqjipSX0EAfp5n0g4qg/edit?usp=sharing) for more details

In [1]:
# load in libraries
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
!pip install xlsxwriter
import xlsxwriter
import numpy as np
!pip install fuzzywuzzy
from fuzzywuzzy import fuzz

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.9/159.9 kB 3.2 MB/s eta 0:00:00


/usr/local/lib/python3.10/dist-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [2]:
# define files
goget_path = '/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/'
parsed_AI_data = goget_path + '/Data sets & reports (O&G)/O&G global materials/Asset Impact Data/Parsed_Filtered_Asset_Impact_Data_2024_12_26_15:58.xlsx'

existing_goget = goget_path + '/GOGET Iterations/Snapshots of GOGET main data set/GOGET oil & gas extraction sites (2024 12 24).xlsx'
existing_goget = pd.read_excel(existing_goget) # This line is added to load the data from the file path.


In [3]:
# check if data was properly imported
parsed_AI_data = pd.read_excel(parsed_AI_data)
parsed_AI_data.head()


,Country/Area,"Subnational unit (province, state)",Field Name,Formation Name,Well ID(s),Longitude,Latitude,Operator,Owner,Resource Type,Recovery mechanism,Onshore/Offshore,Oil production (million bbl/y),Condensate production (million bbl/y),Raw gas production (million m³/y),Dry gas production (million m³/y),NGL production (million bbl/y),Total hydrocarbon production (million bbl/y),Government unit ID
0,United_States,Texas,Spraberry (Trend Area),Trend Area,291857,-101.884138,31.910934,"Pioneer Natural Res. Usa, Inc.","Pioneer Natural Res. Usa, Inc.",Unconventional,Hydraulic fracturing,Onshore,162.669853,0.002624,13897.997906,8668.936792,59.262730,272.934562,85279200_665748
1,United_States,Pennsylvania,NaN,Marcellus,007-20315; 015-20085; 015-20097; 015-20113; 01...,-76.366726,41.649466,Chesapeake Appalachia Llc,NaN,Unconventional,Hydraulic fracturing; Recovery Mechanism Not ...,Onshore,0.000000,0.028151,36901.093126,0.000000,0.000000,217.117282,NaN
2,United_States,Pennsylvania,NaN,Marcellus,115-20007; 115-20008; 115-20010; 115-20011; 11...,-75.834617,41.750138,Coterra Energy Inc,NaN,Unconventional,Recovery Mechanism Not Stated in delivered da...,Onshore,0.000000,0.000000,27529.880422,0.000000,0.000000,161.958287,NaN
3,United_States,Pennsylvania,NaN,Marcellus,003-21982; 003-22193; 003-22194; 003-22195; 00...,-79.914924,40.395492,Range Resources Appalachia Llc,NaN,Unconventional,Recovery Mechanism Not Stated in delivered da...,Onshore,0.000000,3.119199,21699.334162,0.000000,0.000000,130.776382,NaN
4,United_States,Texas,Spraberry (Trend Area),Trend Area,295113,-101.993311,32.211909,Diamondback E&P Llc,Diamondback E&P Llc,Unconventional,Hydraulic fracturing,Onshore,91.013071,0.000000,7190.822048,6661.008955,0.078891,130.278678,85279200_217012


In [4]:
# Filter GOGET to relevant sections - USA, Texas, Pennslyvania and California
relevant_goget = existing_goget[
    (existing_goget['Country/Area'] == 'United States') &
    (existing_goget['Subnational unit (province, state)'].isin(['Texas', 'California', 'Pennsylvania']))
]

In [ ]:
# # prompt: Provide a count of "Government unit ID" matches between relevant_goget and parsed_AI_data

# # Ensure "Government unit ID" column exists in both dataframes
# if "Government unit ID" not in relevant_goget.columns:
#   print("Error: 'Government unit ID' column not found in relevant_goget dataframe.")
#   exit()
# if "Government unit ID" not in parsed_AI_data.columns:
#     print("Error: 'Government unit ID' column not found in parsed_AI_data dataframe.")
#     exit()

# # Count of matches
# matches = 0
# for id_goget in relevant_goget["Government unit ID"].dropna().unique():
#   if id_goget in parsed_AI_data["Government unit ID"].dropna().unique():
#     matches +=1

# print(f"Number of 'Government unit ID' matches between relevant_goget and parsed_AI_data: {matches}")

Number of 'Government unit ID' matches between relevant_goget and parsed_AI_data: 362


In [5]:
# Define correct totals
California_Total = 116.6
Pennsylvania_total = 1204.9
Texas_total = 4140

#Group by state and sum Production Total
production_by_state_1 = parsed_AI_data.groupby('Subnational unit (province, state)')['Total hydrocarbon production (million bbl/y)'].sum()

# Print the result
production_by_state_1

,Total hydrocarbon production (million bbl/y)
"Subnational unit (province, state)",
California,116.630918
Pennsylvania,1204.916532
Texas,4139.981983


In [ ]:
# # prompt: merge relevant_goget's "Discovery year"	"Discovery year source" and "Basin" for all rows where the government ID matches onto parsed_AI_data

# # Merge the dataframes based on "Government unit ID", keeping only parsed_AI_data rows
# merged_data = pd.merge(parsed_AI_data, relevant_goget[['Government unit ID', 'Discovery year', 'Discovery year source', 'Basin']], on='Government unit ID', how='left')

# # Remove duplicate rows based on 'Government unit ID', keeping the first occurrence
# merged_data = merged_data.drop_duplicates(subset=['Government unit ID'], keep='first')

# # Display the first few rows of the merged data
# print(f"Shape of merged_data before deduplication: {merged_data.shape}")

In [8]:
# # Deduplicate merged_data based on all columns.
# deduplicated_merged_data = merged_data.drop_duplicates()

# # Display the shape before and after deduplication
# print(f"Shape of merged_data before deduplication: {merged_data.shape}")
# print(f"Shape of merged_data after deduplication: {deduplicated_merged_data.shape}")

# # You can also deduplicate based on a subset of columns, for example:
# # deduplicated_merged_data = merged_data.drop_duplicates(subset=['Government unit ID'])
# # print(f"Shape of merged_data after deduplication by 'Government unit ID': {deduplicated_merged_data.shape}")

# # optionally: overwrite the previous variable
# merged_data = deduplicated_merged_data
# merged_data.head()

merged_data = parsed_AI_data

In [12]:
#Group by state and sum Production Total
production_by_state_2 = merged_data.groupby('Subnational unit (province, state)')['Total hydrocarbon production (million bbl/y)'].sum()

# check production total still matches
production_by_state_1 == production_by_state_2

,Total hydrocarbon production (million bbl/y)
"Subnational unit (province, state)",
California,True
Pennsylvania,True
Texas,True


In [9]:
# Using assign and lambda functions for a more concise approach
california_goget = (
    relevant_goget[relevant_goget['Subnational unit (province, state)'] == 'California']
    .assign(Field_ID=lambda x: x['Government unit ID'].astype(str).str.split('_').str[0], # Convert to string before using .str
            Field_Name=lambda x: x['Unit Name'].astype(str).str.split(' - ').str[0]) # Convert to string before using .str
    [['Field_Name', 'Field_ID', 'Discovery year', 'Discovery year source']]
)
california_goget


,Field_Name,Field_ID,Discovery year,Discovery year source
14805,Kern River,340,1899.0,EIATop1002015
14806,Belridge South,52,1911.0,EIATop1002015
14807,Belridge South,52,1911.0,EIATop1002015
14808,Belridge South,52,1911.0,EIATop1002015
14809,Midway-Sunset,464,1901.0,EIATop1002015
...,...,...,...,...
15070,Wilmington,849,1932.0,EIATop1002015
15071,Wilmington,849,1932.0,EIATop1002015
15072,Wilmington,849,1932.0,EIATop1002015
15073,Wilmington,849,1932.0,EIATop1002015


In [10]:
# Function to find the best fuzzy match and merge data
def fuzzy_merge_california(row, california_goget):
    if row['Subnational unit (province, state)'] == 'California':
        best_match = None
        best_ratio = 0
        for index, cali_row in california_goget.iterrows():
            ratio = fuzz.ratio(row['Field Name'], cali_row['Field_Name'])
            if ratio > best_ratio:
                best_ratio = ratio
                best_match = cali_row

        # Check if a good match was found (e.g., above a certain threshold)
        if best_ratio > 70:  # Adjust threshold as needed
            return pd.Series([best_match['Discovery year'], best_match['Discovery year source']])
    return pd.Series([None, None])

# Apply the function to create new columns
new_columns = merged_data.apply(fuzzy_merge_california, axis=1, args=(california_goget,))
merged_data['California Discovery year'] = new_columns[0]
merged_data['California Discovery year source'] = new_columns[1]


In [13]:
#Group by state and sum Production Total
production_by_state_3 = merged_data.groupby('Subnational unit (province, state)')['Total hydrocarbon production (million bbl/y)'].sum()

# check if production still matches
production_by_state_3 == production_by_state_2

,Total hydrocarbon production (million bbl/y)
"Subnational unit (province, state)",
California,True
Pennsylvania,True
Texas,True


In [14]:
# Replace "Discovery year" and "Discovery year source" with "California Discovery year" and "California Discovery year source" for California rows
merged_data.loc[merged_data['Subnational unit (province, state)'] == 'California', 'Discovery year'] = merged_data['California Discovery year']
merged_data.loc[merged_data['Subnational unit (province, state)'] == 'California', 'Discovery year source'] = merged_data['California Discovery year source']

# show california rows of merged_data
merged_data[merged_data['Subnational unit (province, state)'] == 'California']

,Country/Area,"Subnational unit (province, state)",Field Name,Formation Name,Well ID(s),Longitude,Latitude,Operator,Owner,Resource Type,...,Condensate production (million bbl/y),Raw gas production (million m³/y),Dry gas production (million m³/y),NGL production (million bbl/y),Total hydrocarbon production (million bbl/y),Government unit ID,California Discovery year,California Discovery year source,Discovery year,Discovery year source
82,United_States,California,Kern River,Kern River,4.0301e+10; 40290000100; 40290003800; 40290003...,-118.988572,35.447675,Chevron U.S.A. Inc.,Owner Not Stated In Delivered Dataset,Unconventional,...,0.0,11.797519,0.0,0.0,13.873847,NaN,1899.0,EIATop1002015,1899.0,EIATop1002015
191,United_States,California,Belridge South,Tulare,40294598600; 40294713700; 40295201100; 4029534...,-119.700655,35.440810,Aera Energy Llc,Owner Not Stated In Delivered Dataset,Unconventional,...,0.0,39.378744,0.0,0.0,4.707407,NaN,1911.0,EIATop1002015,1911.0,EIATop1002015
199,United_States,California,Belridge South,Diatomite,40298229800; 40298481500; 40298588000; 4030002...,-119.719752,35.456440,Aera Energy Llc,Owner Not Stated In Delivered Dataset,Unconventional,...,0.0,48.146666,0.0,0.0,4.413649,NaN,1911.0,EIATop1002015,1911.0,EIATop1002015
222,United_States,California,Belridge South,Diatomite,40295932100; 40296205100; 40296205400; 4029644...,-119.724851,35.458641,Aera Energy Llc,Owner Not Stated In Delivered Dataset,Conventional,...,0.0,50.120323,0.0,0.0,3.659207,NaN,1911.0,EIATop1002015,1911.0,EIATop1002015
225,United_States,California,Midway-Sunset,No Formation Stated In Delivered Data,40291195600; 40291195900; 40291198000; 4029439...,-119.498174,35.166537,Berry Petroleum Company Llc,Owner Not Stated In Delivered Dataset,Unconventional,...,0.0,0.000000,0.0,0.0,3.615242,NaN,1901.0,EIATop1002015,1901.0,EIATop1002015
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1157,United_States,California,Edison,No Formation Stated In Delivered Data,40290614100; 40290614200; 40290616700; 4029527...,-118.836857,35.315924,Hunter Edison Oil Development Limited Partnership,Owner Not Stated In Delivered Dataset,Unconventional,...,0.0,0.262019,0.0,0.0,0.037256,NaN,NaN,NaN,NaN,NaN
1158,United_States,California,Torrance,Others,40371659700; 40371659701; 40371659800; 4037165...,-118.282589,33.796522,E & B Natural Resources Management Corporation,Owner Not Stated In Delivered Dataset,Conventional,...,0.0,0.173611,0.0,0.0,0.036795,NaN,NaN,NaN,NaN,NaN
1159,United_States,California,Las Cienegas,No Formation Stated In Delivered Data,40370032200; 40370036900; 40370037200; 4037003...,-118.312648,34.032264,E & B Natural Resources Management Corporation,Owner Not Stated In Delivered Dataset,Unconventional,...,0.0,0.493025,0.0,0.0,0.036617,NaN,NaN,NaN,NaN,NaN
1160,United_States,California,Kern Front,Chanac,40305996700; 40306059100; 40306082900; 4030608...,-119.034385,35.462741,California Resources Production Corporation,Owner Not Stated In Delivered Dataset,Unconventional,...,0.0,0.000000,0.0,0.0,0.036237,NaN,1925.0,EIATop1002015,1925.0,EIATop1002015


In [15]:
# drop 'California Discovery year' and 'California Discovery year source' from merged_data
merged_data = merged_data.drop(columns=['California Discovery year', 'California Discovery year source'])
merged_data.head()

,Country/Area,"Subnational unit (province, state)",Field Name,Formation Name,Well ID(s),Longitude,Latitude,Operator,Owner,Resource Type,...,Onshore/Offshore,Oil production (million bbl/y),Condensate production (million bbl/y),Raw gas production (million m³/y),Dry gas production (million m³/y),NGL production (million bbl/y),Total hydrocarbon production (million bbl/y),Government unit ID,Discovery year,Discovery year source
0,United_States,Texas,Spraberry (Trend Area),Trend Area,291857,-101.884138,31.910934,"Pioneer Natural Res. Usa, Inc.","Pioneer Natural Res. Usa, Inc.",Unconventional,...,Onshore,162.669853,0.002624,13897.997906,8668.936792,59.262730,272.934562,85279200_665748,NaN,NaN
1,United_States,Pennsylvania,NaN,Marcellus,007-20315; 015-20085; 015-20097; 015-20113; 01...,-76.366726,41.649466,Chesapeake Appalachia Llc,NaN,Unconventional,...,Onshore,0.000000,0.028151,36901.093126,0.000000,0.000000,217.117282,NaN,NaN,NaN
2,United_States,Pennsylvania,NaN,Marcellus,115-20007; 115-20008; 115-20010; 115-20011; 11...,-75.834617,41.750138,Coterra Energy Inc,NaN,Unconventional,...,Onshore,0.000000,0.000000,27529.880422,0.000000,0.000000,161.958287,NaN,NaN,NaN
3,United_States,Pennsylvania,NaN,Marcellus,003-21982; 003-22193; 003-22194; 003-22195; 00...,-79.914924,40.395492,Range Resources Appalachia Llc,NaN,Unconventional,...,Onshore,0.000000,3.119199,21699.334162,0.000000,0.000000,130.776382,NaN,NaN,NaN
4,United_States,Texas,Spraberry (Trend Area),Trend Area,295113,-101.993311,32.211909,Diamondback E&P Llc,Diamondback E&P Llc,Unconventional,...,Onshore,91.013071,0.000000,7190.822048,6661.008955,0.078891,130.278678,85279200_217012,NaN,NaN


In [16]:
# format Discovery year as four character from the left
merged_data['Discovery year'] = merged_data['Discovery year'].astype(str)
merged_data['Discovery year'] = merged_data['Discovery year'].str[:4]
merged_data.head()

,Country/Area,"Subnational unit (province, state)",Field Name,Formation Name,Well ID(s),Longitude,Latitude,Operator,Owner,Resource Type,...,Onshore/Offshore,Oil production (million bbl/y),Condensate production (million bbl/y),Raw gas production (million m³/y),Dry gas production (million m³/y),NGL production (million bbl/y),Total hydrocarbon production (million bbl/y),Government unit ID,Discovery year,Discovery year source
0,United_States,Texas,Spraberry (Trend Area),Trend Area,291857,-101.884138,31.910934,"Pioneer Natural Res. Usa, Inc.","Pioneer Natural Res. Usa, Inc.",Unconventional,...,Onshore,162.669853,0.002624,13897.997906,8668.936792,59.262730,272.934562,85279200_665748,nan,NaN
1,United_States,Pennsylvania,NaN,Marcellus,007-20315; 015-20085; 015-20097; 015-20113; 01...,-76.366726,41.649466,Chesapeake Appalachia Llc,NaN,Unconventional,...,Onshore,0.000000,0.028151,36901.093126,0.000000,0.000000,217.117282,NaN,nan,NaN
2,United_States,Pennsylvania,NaN,Marcellus,115-20007; 115-20008; 115-20010; 115-20011; 11...,-75.834617,41.750138,Coterra Energy Inc,NaN,Unconventional,...,Onshore,0.000000,0.000000,27529.880422,0.000000,0.000000,161.958287,NaN,nan,NaN
3,United_States,Pennsylvania,NaN,Marcellus,003-21982; 003-22193; 003-22194; 003-22195; 00...,-79.914924,40.395492,Range Resources Appalachia Llc,NaN,Unconventional,...,Onshore,0.000000,3.119199,21699.334162,0.000000,0.000000,130.776382,NaN,nan,NaN
4,United_States,Texas,Spraberry (Trend Area),Trend Area,295113,-101.993311,32.211909,Diamondback E&P Llc,Diamondback E&P Llc,Unconventional,...,Onshore,91.013071,0.000000,7190.822048,6661.008955,0.078891,130.278678,85279200_217012,nan,NaN


In [17]:
#Group by state and sum Production Total
production_by_state_4 = merged_data.groupby('Subnational unit (province, state)')['Total hydrocarbon production (million bbl/y)'].sum()

# Print the result
production_by_state_4 == production_by_state_1

,Total hydrocarbon production (million bbl/y)
"Subnational unit (province, state)",
California,True
Pennsylvania,True
Texas,True


In [18]:
# seperate out PA to create ID seperately
merged_data_noPA = merged_data[merged_data['Subnational unit (province, state)'] != 'Pennsylvania']
merged_data_PA = merged_data[merged_data['Subnational unit (province, state)'] == 'Pennsylvania']


In [19]:
def create_unit_name_and_tag_duplicates(df, state_specific_column):
    """
    Creates a 'Unit Name' column and tags duplicates.

    Args:
        df: The input DataFrame (merged_data_noPA or merged_data_PA).
        state_specific_column: The column to use for Unit Name creation
                               ('Field Name' for TX/CA, 'Formation Name' for PA).

    Returns:
        The modified DataFrame with 'Unit Name' and 'Duplicate' columns.
    """
    df['Unit Name'] = df[state_specific_column] + ' - ' + df['Operator']
    df['Duplicate'] = df['Unit Name'].duplicated(keep=False)
    return df

# Apply the function to both DataFrames
merged_data_noPA = create_unit_name_and_tag_duplicates(merged_data_noPA, 'Field Name')
merged_data_PA = create_unit_name_and_tag_duplicates(merged_data_PA, 'Formation Name')

merged_data_PA.head()

<ipython-input-19-0a34c484bb10>:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Unit Name'] = df[state_specific_column] + ' - ' + df['Operator']
<ipython-input-19-0a34c484bb10>:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Duplicate'] = df['Unit Name'].duplicated(keep=False)
<ipython-input-19-0a34c484bb10>:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation:

,Country/Area,"Subnational unit (province, state)",Field Name,Formation Name,Well ID(s),Longitude,Latitude,Operator,Owner,Resource Type,...,Condensate production (million bbl/y),Raw gas production (million m³/y),Dry gas production (million m³/y),NGL production (million bbl/y),Total hydrocarbon production (million bbl/y),Government unit ID,Discovery year,Discovery year source,Unit Name,Duplicate
1,United_States,Pennsylvania,NaN,Marcellus,007-20315; 015-20085; 015-20097; 015-20113; 01...,-76.366726,41.649466,Chesapeake Appalachia Llc,NaN,Unconventional,...,0.028151,36901.093126,0.0,0.0,217.117282,NaN,nan,NaN,Marcellus - Chesapeake Appalachia Llc,False
2,United_States,Pennsylvania,NaN,Marcellus,115-20007; 115-20008; 115-20010; 115-20011; 11...,-75.834617,41.750138,Coterra Energy Inc,NaN,Unconventional,...,0.000000,27529.880422,0.0,0.0,161.958287,NaN,nan,NaN,Marcellus - Coterra Energy Inc,False
3,United_States,Pennsylvania,NaN,Marcellus,003-21982; 003-22193; 003-22194; 003-22195; 00...,-79.914924,40.395492,Range Resources Appalachia Llc,NaN,Unconventional,...,3.119199,21699.334162,0.0,0.0,130.776382,NaN,nan,NaN,Marcellus - Range Resources Appalachia Llc,False
7,United_States,Pennsylvania,NaN,Marcellus,003-22234; 003-22235; 003-22236; 003-22237; 00...,-79.946551,40.148889,Eqt Prod Co,NaN,Unconventional,...,0.023174,17447.036388,0.0,0.0,102.664089,NaN,nan,NaN,Marcellus - Eqt Prod Co,False
8,United_States,Pennsylvania,NaN,Marcellus,059-25647; 059-25669; 059-25685; 059-25739; 05...,-80.207659,39.977252,Rice Drilling B Llc,NaN,Unconventional,...,0.000000,14241.750745,0.0,0.0,83.784220,NaN,nan,NaN,Marcellus - Rice Drilling B Llc,False


In [20]:
# prompt: merge merged_data_noPA back with merged_data_PA

# Concatenate the two dataframes back together
merged_data = pd.concat([merged_data_noPA, merged_data_PA], ignore_index=True)

# Display the first few rows of the merged data
merged_data.head()

,Country/Area,"Subnational unit (province, state)",Field Name,Formation Name,Well ID(s),Longitude,Latitude,Operator,Owner,Resource Type,...,Condensate production (million bbl/y),Raw gas production (million m³/y),Dry gas production (million m³/y),NGL production (million bbl/y),Total hydrocarbon production (million bbl/y),Government unit ID,Discovery year,Discovery year source,Unit Name,Duplicate
0,United_States,Texas,Spraberry (Trend Area),Trend Area,291857,-101.884138,31.910934,"Pioneer Natural Res. Usa, Inc.","Pioneer Natural Res. Usa, Inc.",Unconventional,...,0.002624,13897.997906,8668.936792,59.262730,272.934562,85279200_665748,nan,NaN,Spraberry (Trend Area) - Pioneer Natural Res. ...,True
1,United_States,Texas,Spraberry (Trend Area),Trend Area,295113,-101.993311,32.211909,Diamondback E&P Llc,Diamondback E&P Llc,Unconventional,...,0.000000,7190.822048,6661.008955,0.078891,130.278678,85279200_217012,nan,NaN,Spraberry (Trend Area) - Diamondback E&P Llc,True
2,United_States,Texas,Spraberry (Trend Area),Trend Area,NaN,-101.845793,31.996353,Endeavor Energy Resources L.P.,Endeavor Energy Resources L.P.,Unconventional,...,0.000000,6615.620810,4107.112929,27.966809,127.701538,85279200_251726,nan,NaN,Spraberry (Trend Area) - Endeavor Energy Resou...,True
3,United_States,Texas,Phantom (Wolfcamp),Wolfcamp,276220; 276320; 276322; 276516; 276545; 276563...,-103.669860,31.779745,Anadarko E&P Onshore Llc,Anadarko E&P Onshore Llc,Unconventional,...,28.699308,7341.338037,4173.713319,27.539786,119.184498,71052900_20528,nan,NaN,Phantom (Wolfcamp) - Anadarko E&P Onshore Llc,True
4,United_States,Texas,Spraberry (Trend Area),Trend Area,NaN,-101.899898,31.878125,Xto Energy Inc.,Xto Energy Inc.,Unconventional,...,0.000000,4775.216863,2999.418387,20.504665,75.794375,85279200_945936,nan,NaN,Spraberry (Trend Area) - Xto Energy Inc.,True


In [21]:
#Group by state and sum Production Total
production_by_state_5 = merged_data.groupby('Subnational unit (province, state)')['Total hydrocarbon production (million bbl/y)'].sum()

# Print the result
production_by_state_5 == production_by_state_1

,Total hydrocarbon production (million bbl/y)
"Subnational unit (province, state)",
California,True
Pennsylvania,True
Texas,True


In [22]:
# clean Unit Names
# List of terms to remove
terms_to_remove = [" Inc.", " LLC", " Llc", " L.P.", " L.L.C.", "Ltd.", " LP", " Corp", " Ltd", " Lp",  " Incorporated", " Limited Liability Company", " Limited Partnership", " Corporation"," Company", ", "]

# Function to strip the terms from the Unit Name
def strip_terms(unit_name):
    if isinstance(unit_name, str):  # Check if the unit name is a string
        for term in terms_to_remove:
            unit_name = unit_name.replace(term, "")
        return unit_name.strip()  # Remove leading/trailing whitespace
    else:
        return unit_name

# Apply the function to the 'Unit Name' column
merged_data['Unit Name'] = merged_data['Unit Name'].apply(strip_terms)


merged_data.head()

,Country/Area,"Subnational unit (province, state)",Field Name,Formation Name,Well ID(s),Longitude,Latitude,Operator,Owner,Resource Type,...,Condensate production (million bbl/y),Raw gas production (million m³/y),Dry gas production (million m³/y),NGL production (million bbl/y),Total hydrocarbon production (million bbl/y),Government unit ID,Discovery year,Discovery year source,Unit Name,Duplicate
0,United_States,Texas,Spraberry (Trend Area),Trend Area,291857,-101.884138,31.910934,"Pioneer Natural Res. Usa, Inc.","Pioneer Natural Res. Usa, Inc.",Unconventional,...,0.002624,13897.997906,8668.936792,59.262730,272.934562,85279200_665748,nan,NaN,Spraberry (Trend Area) - Pioneer Natural Res. ...,True
1,United_States,Texas,Spraberry (Trend Area),Trend Area,295113,-101.993311,32.211909,Diamondback E&P Llc,Diamondback E&P Llc,Unconventional,...,0.000000,7190.822048,6661.008955,0.078891,130.278678,85279200_217012,nan,NaN,Spraberry (Trend Area) - Diamondback E&P,True
2,United_States,Texas,Spraberry (Trend Area),Trend Area,NaN,-101.845793,31.996353,Endeavor Energy Resources L.P.,Endeavor Energy Resources L.P.,Unconventional,...,0.000000,6615.620810,4107.112929,27.966809,127.701538,85279200_251726,nan,NaN,Spraberry (Trend Area) - Endeavor Energy Resou...,True
3,United_States,Texas,Phantom (Wolfcamp),Wolfcamp,276220; 276320; 276322; 276516; 276545; 276563...,-103.669860,31.779745,Anadarko E&P Onshore Llc,Anadarko E&P Onshore Llc,Unconventional,...,28.699308,7341.338037,4173.713319,27.539786,119.184498,71052900_20528,nan,NaN,Phantom (Wolfcamp) - Anadarko E&P Onshore,True
4,United_States,Texas,Spraberry (Trend Area),Trend Area,NaN,-101.899898,31.878125,Xto Energy Inc.,Xto Energy Inc.,Unconventional,...,0.000000,4775.216863,2999.418387,20.504665,75.794375,85279200_945936,nan,NaN,Spraberry (Trend Area) - Xto Energy,True


In [23]:
# replace "United_States" with United States
merged_data['Country/Area'] = merged_data['Country/Area'].replace('United_States', 'United States')
merged_data.head()

,Country/Area,"Subnational unit (province, state)",Field Name,Formation Name,Well ID(s),Longitude,Latitude,Operator,Owner,Resource Type,...,Condensate production (million bbl/y),Raw gas production (million m³/y),Dry gas production (million m³/y),NGL production (million bbl/y),Total hydrocarbon production (million bbl/y),Government unit ID,Discovery year,Discovery year source,Unit Name,Duplicate
0,United States,Texas,Spraberry (Trend Area),Trend Area,291857,-101.884138,31.910934,"Pioneer Natural Res. Usa, Inc.","Pioneer Natural Res. Usa, Inc.",Unconventional,...,0.002624,13897.997906,8668.936792,59.262730,272.934562,85279200_665748,nan,NaN,Spraberry (Trend Area) - Pioneer Natural Res. ...,True
1,United States,Texas,Spraberry (Trend Area),Trend Area,295113,-101.993311,32.211909,Diamondback E&P Llc,Diamondback E&P Llc,Unconventional,...,0.000000,7190.822048,6661.008955,0.078891,130.278678,85279200_217012,nan,NaN,Spraberry (Trend Area) - Diamondback E&P,True
2,United States,Texas,Spraberry (Trend Area),Trend Area,NaN,-101.845793,31.996353,Endeavor Energy Resources L.P.,Endeavor Energy Resources L.P.,Unconventional,...,0.000000,6615.620810,4107.112929,27.966809,127.701538,85279200_251726,nan,NaN,Spraberry (Trend Area) - Endeavor Energy Resou...,True
3,United States,Texas,Phantom (Wolfcamp),Wolfcamp,276220; 276320; 276322; 276516; 276545; 276563...,-103.669860,31.779745,Anadarko E&P Onshore Llc,Anadarko E&P Onshore Llc,Unconventional,...,28.699308,7341.338037,4173.713319,27.539786,119.184498,71052900_20528,nan,NaN,Phantom (Wolfcamp) - Anadarko E&P Onshore,True
4,United States,Texas,Spraberry (Trend Area),Trend Area,NaN,-101.899898,31.878125,Xto Energy Inc.,Xto Energy Inc.,Unconventional,...,0.000000,4775.216863,2999.418387,20.504665,75.794375,85279200_945936,nan,NaN,Spraberry (Trend Area) - Xto Energy,True


In [24]:
#Group by state and sum Production Total
production_by_state_6 = merged_data.groupby('Subnational unit (province, state)')['Total hydrocarbon production (million bbl/y)'].sum()

# Print the result
production_by_state_6 == production_by_state_1

,Total hydrocarbon production (million bbl/y)
"Subnational unit (province, state)",
California,True
Pennsylvania,True
Texas,True


In [25]:
# Create new columns and initialize with None
merged_data[['Status', 'Status year', 'Status source', 'Location accuracy', 'Location source']] = None

# Fill in 'Status' related columns based on hydrocarbon production
mask_production = merged_data['Total hydrocarbon production (million bbl/y)'] > 0
merged_data.loc[mask_production, ['Status', 'Status year', 'Status source']] = ['operating', 2023, 'AIProduction2023']

# Fill in 'Location' related columns based on latitude availability
mask_latitude = merged_data['Latitude'].notna()
merged_data.loc[mask_latitude, ['Location accuracy', 'Location source']] = ['exact', 'AIlocation2023']

merged_data.head()

,Country/Area,"Subnational unit (province, state)",Field Name,Formation Name,Well ID(s),Longitude,Latitude,Operator,Owner,Resource Type,...,Government unit ID,Discovery year,Discovery year source,Unit Name,Duplicate,Status,Status year,Status source,Location accuracy,Location source
0,United States,Texas,Spraberry (Trend Area),Trend Area,291857,-101.884138,31.910934,"Pioneer Natural Res. Usa, Inc.","Pioneer Natural Res. Usa, Inc.",Unconventional,...,85279200_665748,nan,NaN,Spraberry (Trend Area) - Pioneer Natural Res. ...,True,operating,2023,AIProduction2023,exact,AIlocation2023
1,United States,Texas,Spraberry (Trend Area),Trend Area,295113,-101.993311,32.211909,Diamondback E&P Llc,Diamondback E&P Llc,Unconventional,...,85279200_217012,nan,NaN,Spraberry (Trend Area) - Diamondback E&P,True,operating,2023,AIProduction2023,exact,AIlocation2023
2,United States,Texas,Spraberry (Trend Area),Trend Area,NaN,-101.845793,31.996353,Endeavor Energy Resources L.P.,Endeavor Energy Resources L.P.,Unconventional,...,85279200_251726,nan,NaN,Spraberry (Trend Area) - Endeavor Energy Resou...,True,operating,2023,AIProduction2023,exact,AIlocation2023
3,United States,Texas,Phantom (Wolfcamp),Wolfcamp,276220; 276320; 276322; 276516; 276545; 276563...,-103.669860,31.779745,Anadarko E&P Onshore Llc,Anadarko E&P Onshore Llc,Unconventional,...,71052900_20528,nan,NaN,Phantom (Wolfcamp) - Anadarko E&P Onshore,True,operating,2023,AIProduction2023,exact,AIlocation2023
4,United States,Texas,Spraberry (Trend Area),Trend Area,NaN,-101.899898,31.878125,Xto Energy Inc.,Xto Energy Inc.,Unconventional,...,85279200_945936,nan,NaN,Spraberry (Trend Area) - Xto Energy,True,operating,2023,AIProduction2023,exact,AIlocation2023


In [26]:
# Add specicifity to sources

# Append Subnational unit to Status source and Location source
merged_data['Status source'] = merged_data.apply(lambda row: f"{row['Status source']} ({row['Subnational unit (province, state)']})" if pd.notnull(row['Status source']) else None, axis=1)
merged_data['Location source'] = merged_data.apply(lambda row: f"{row['Location source']} ({row['Subnational unit (province, state)']})" if pd.notnull(row['Location source']) else None, axis=1)


merged_data.head()

,Country/Area,"Subnational unit (province, state)",Field Name,Formation Name,Well ID(s),Longitude,Latitude,Operator,Owner,Resource Type,...,Government unit ID,Discovery year,Discovery year source,Unit Name,Duplicate,Status,Status year,Status source,Location accuracy,Location source
0,United States,Texas,Spraberry (Trend Area),Trend Area,291857,-101.884138,31.910934,"Pioneer Natural Res. Usa, Inc.","Pioneer Natural Res. Usa, Inc.",Unconventional,...,85279200_665748,nan,NaN,Spraberry (Trend Area) - Pioneer Natural Res. ...,True,operating,2023,AIProduction2023 (Texas),exact,AIlocation2023 (Texas)
1,United States,Texas,Spraberry (Trend Area),Trend Area,295113,-101.993311,32.211909,Diamondback E&P Llc,Diamondback E&P Llc,Unconventional,...,85279200_217012,nan,NaN,Spraberry (Trend Area) - Diamondback E&P,True,operating,2023,AIProduction2023 (Texas),exact,AIlocation2023 (Texas)
2,United States,Texas,Spraberry (Trend Area),Trend Area,NaN,-101.845793,31.996353,Endeavor Energy Resources L.P.,Endeavor Energy Resources L.P.,Unconventional,...,85279200_251726,nan,NaN,Spraberry (Trend Area) - Endeavor Energy Resou...,True,operating,2023,AIProduction2023 (Texas),exact,AIlocation2023 (Texas)
3,United States,Texas,Phantom (Wolfcamp),Wolfcamp,276220; 276320; 276322; 276516; 276545; 276563...,-103.669860,31.779745,Anadarko E&P Onshore Llc,Anadarko E&P Onshore Llc,Unconventional,...,71052900_20528,nan,NaN,Phantom (Wolfcamp) - Anadarko E&P Onshore,True,operating,2023,AIProduction2023 (Texas),exact,AIlocation2023 (Texas)
4,United States,Texas,Spraberry (Trend Area),Trend Area,NaN,-101.899898,31.878125,Xto Energy Inc.,Xto Energy Inc.,Unconventional,...,85279200_945936,nan,NaN,Spraberry (Trend Area) - Xto Energy,True,operating,2023,AIProduction2023 (Texas),exact,AIlocation2023 (Texas)


In [27]:
merged_data.columns

Index(['Country/Area', 'Subnational unit (province, state)', 'Field Name',
       'Formation Name', 'Well ID(s)', 'Longitude', 'Latitude', 'Operator',
       'Owner', 'Resource Type', 'Recovery mechanism', 'Onshore/Offshore',
       'Oil production (million bbl/y)',
       'Condensate production (million bbl/y)',
       'Raw gas production (million m³/y)',
       'Dry gas production (million m³/y)', 'NGL production (million bbl/y)',
       'Total hydrocarbon production (million bbl/y)', 'Government unit ID',
       'Discovery year', 'Discovery year source', 'Unit Name', 'Duplicate',
       'Status', 'Status year', 'Status source', 'Location accuracy',
       'Location source'],
      dtype='object')

In [28]:
# show merged_data columns
merged_data_clean = merged_data.rename(columns={
    'Country/Area': 'Country/Area',
    'Subnational unit (province, state)': 'Subnational unit (province, state)',
    'Unit Name': 'Unit Name',  # generated name of field name plus operator for tx and ca, formation name plus operator in pa
    'Field Name': 'Sub Formation Name',
    'Formation Name': 'Project Name',  # for summing by formation
    'Status': 'Status',
    'Status year': 'Status year',
    'Status source': 'Status source',
    'Location accuracy': 'Location accuracy',
    'Location source': 'Location source',
    'Well ID(s)': 'Well ID(s)',
    'Latitude': 'Latitude',
    'Longitude': 'Longitude',
    'Operator': 'Operator',
    'Owner': 'Owner',
    'Resource Type': 'Resource Type',
    'Recovery mechanism': 'Recovery mechanism',
    'Onshore/Offshore': 'Onshore/Offshore',
    'Oil production (million bbl/y)': 'Oil production (million bbl/y)',
    'Condensate production (million bbl/y)': 'Condensate production (million bbl/y)',
    'Raw gas production (million m³/y)': 'Raw gas production (million m³/y)',
    'Dry gas production (million m³/y)': 'Dry gas production (million m³/y)',
    'NGL production (million bbl/y)': 'NGL production (million bbl/y)',
    'Total hydrocarbon production (million bbl/y)': 'Total hydrocarbon production (million boe/y)',
    'Government unit ID': 'Government unit ID',
    #'Discovery year': 'Discovery year',
    #'Discovery year source': 'Discovery year source',
    'Duplicate': 'Duplicate'
})

# reorder merged_data_clean
merged_data_clean_reorder = ['Unit Name', 'Country/Area', 'Subnational unit (province, state)', 'Status', 'Status year', 'Status source', 'Latitude', 'Longitude', 'Location accuracy', 'Location source', 'Operator', 'Owner','Project Name', 'Government unit ID', 'Well ID(s)', 'Onshore/Offshore', 'Resource Type', 'Recovery mechanism', 'Oil production (million bbl/y)', 'Condensate production (million bbl/y)', 'Raw gas production (million m³/y)', 'Dry gas production (million m³/y)', 'NGL production (million bbl/y)', 'Total hydrocarbon production (million boe/y)', "Duplicate"]
# Explicitly define the desired column order

merged_data_clean = merged_data_clean[merged_data_clean_reorder]

# show merged_data_clean
merged_data_clean.columns

Index(['Unit Name', 'Country/Area', 'Subnational unit (province, state)',
       'Status', 'Status year', 'Status source', 'Latitude', 'Longitude',
       'Location accuracy', 'Location source', 'Operator', 'Owner',
       'Project Name', 'Government unit ID', 'Well ID(s)', 'Onshore/Offshore',
       'Resource Type', 'Recovery mechanism', 'Oil production (million bbl/y)',
       'Condensate production (million bbl/y)',
       'Raw gas production (million m³/y)',
       'Dry gas production (million m³/y)', 'NGL production (million bbl/y)',
       'Total hydrocarbon production (million boe/y)', 'Duplicate'],
      dtype='object')

In [29]:
# enure production_by_state totals match

#Group by state and sum Production Total
production_by_state_6 = merged_data_clean.groupby('Subnational unit (province, state)')['Total hydrocarbon production (million boe/y)'].sum()

# Print the result
production_by_state_1 == production_by_state_6

,0
"Subnational unit (province, state)",
California,True
Pennsylvania,True
Texas,True


In [30]:
# # if "Unit Name" is a duplicate, add "Resource Type" + "_" + "Recovery mechanism" to the unit name
# for unit_name in merged_data_clean['Unit Name'].unique():
#     if merged_data_clean['Unit Name'].value_counts()[unit_name] > 1:  # Check if unit name is non-unique
#         # Get the Resource Type and Recovery mechanism for rows with the non-unique unit name
#         resource_types = merged_data_clean.loc[merged_data['Unit Name'] == unit_name, 'Resource Type'].astype(str)
#         recovery_mechanisms = merged_data_clean.loc[merged_data['Unit Name'] == unit_name, 'Recovery mechanism'].astype(str)
#         project_name = merged_data_clean.loc[merged_data['Unit Name'] == unit_name, 'Project Name'].astype(str)

#         # Update the Unit Name for rows with the non-unique unit name
#         merged_data_clean.loc[merged_data_clean['Unit Name'] == unit_name, 'Unit Name'] = (
#             merged_data_clean.loc[merged_data_clean['Unit Name'] == unit_name, 'Unit Name'] +
#             " (" +
#             project_name +
#             ", " +
#             resource_types +
#             "_" +
#             recovery_mechanisms +
#             ")"

#         )

# Count non-unique unit names
num_non_unique_unit_names = merged_data_clean['Unit Name'].duplicated(keep=False).sum()
print(f"Number of non-unique Unit Names: {num_non_unique_unit_names}")

Number of non-unique Unit Names: 443


In [31]:
# Function to modify Unit Name based on Onshore/Offshore and Resource Type
def modify_unit_name(row, df):
    unit_name = row['Unit Name']
    if df['Unit Name'].value_counts()[unit_name] > 1:  # Check if unit name is non-unique

        subset = df[df['Unit Name'] == unit_name]

        if len(subset['Onshore/Offshore'].unique()) > 1:
            if row['Onshore/Offshore'] == 'Onshore':
                unit_name += ' (Onshore)'
            elif row['Onshore/Offshore'] == 'Offshore':
                unit_name += ' (Offshore)'
        elif len(subset['Resource Type'].unique()) > 1:
            if row['Resource Type'] == 'Unconventional':
                unit_name += ' (Unconventional)'
            elif row['Resource Type'] == 'Conventional':
                unit_name += ' (Conventional)'
         # Modified project name handling
        elif len(subset['Project Name'].unique()) > 1:
            unit_name += ' (' + row['Project Name'] + ')' # appends project name
    return unit_name

# Apply the function to modify Unit Names
merged_data_clean['Unit Name'] = merged_data_clean.apply(lambda row: modify_unit_name(row, merged_data_clean), axis=1)

In [32]:
# Recheck non-unique names
num_non_unique_unit_names = merged_data_clean['Unit Name'].duplicated(keep=False).sum()

# update duplicate tag
merged_data_clean['Duplicate'] = merged_data_clean['Unit Name'].duplicated(keep=False)

print(f"Number of non-unique Unit Names: {num_non_unique_unit_names}")

Number of non-unique Unit Names: 124


In [33]:
import datetime
current_datetime = datetime.datetime.now()
formatted_datetime = current_datetime.strftime("%Y_%m_%d_%H:%M")

output_file_path = goget_path + '/Data sets & reports (O&G)/O&G global materials/Asset Impact Data/'
output_file_name = f"formatted_Asset_Impact_Data_{formatted_datetime}.xlsx"
Full_output_file_path = output_file_path + output_file_name

# Create a new ExcelWriter object
writer = pd.ExcelWriter(Full_output_file_path, engine='xlsxwriter') # Create writer with path

# Create the "About" sheet
about_df = pd.DataFrame({
    'Information': [
        f"Generated on: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"Source file: /Data sets & reports (O&G)/O&G global materials/Asset Impact Data/Parsed_Filtered_Asset_Impact_Data_2024_12_18_19:32.xlsx'",
        f"Goget data: /GOGET Iterations/Snapshots of GOGET main data set/GOGET oil & gas extraction sites (2024 12 18 PM).xlsx"
            ]
})
about_df.to_excel(writer, sheet_name='About', index=False)

merged_data_clean.to_excel(writer, sheet_name='Data', index=False)

# Save the Excel file (only once)
writer.close()


In [34]:
# print saved location
print(f"The file has been saved to: {Full_output_file_path}")

The file has been saved to: /content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)//Data sets & reports (O&G)/O&G global materials/Asset Impact Data/formatted_Asset_Impact_Data_2025_01_15_15:29.xlsx
